# BrainWear 2D — Baseline vs Slot+AA-CBR comparison

Builds the **Baseline vs Slot+AA-CBR** comparison for the BrainWear 2D dataset, across
EORTC outcome scores (QL2, PF2, CF, EF) and class granularities k ∈ {2, 3, 4, 5}.

**Pipeline:**
1. Find the best 2D baseline config per outcome score from `baseline/brainwear_leaderboard.json`  
   (only k=3 runs exist there).
2. Re-run those configs for k=2,3,4,5 via `brainwear_2d_baseline_cv.py`  
   (GPU training, ~30-60 min/config on HPC; skipped if already logged).  
   Results go to `baseline/brainwear_baseline_cv.json`.
3. Load Slot+AA-CBR results from `aacbr/leaderboard_trained.json`  
   (already has matched CV for k=2-5; nothing to recompute).
4. Assemble, rank scores by Slot+AA-CBR skill at k=3, show top-3 figure + table.

Run cells top to bottom. Re-running is safe — Cell 3 skips already-logged entries.

In [ ]:
import importlib, json, sys
from pathlib import Path

FYP_ROOT = Path('/path/to/BrainWear_Kareem/FYP')
if str(FYP_ROOT) not in sys.path:
    sys.path.insert(0, str(FYP_ROOT))

from baseline import brainwear_2d_baseline_cv as bcv
from comparison import compare_brainwear as cmp_bw
importlib.reload(bcv); importlib.reload(cmp_bw)

SEED, N_FOLDS = 0, 5
HYPER_LB       = FYP_ROOT / 'baseline' / 'leaderboard.json'           # hyperparameter sweep
BASELINE_LB    = FYP_ROOT / 'baseline' / 'brainwear_leaderboard.json' # outcome sweep (per-score)
BASELINE_CV_LB = FYP_ROOT / 'baseline' / 'brainwear_baseline_cv.json'
TRAINED_LB     = FYP_ROOT / 'aacbr'    / 'leaderboard_trained.json'
EVAL_SCRIPT    = 'eval_trained_brainwear_2d'

print('Setup OK.')
print(f'Hyper leaderboard exists:    {HYPER_LB.exists()}')
print(f'Baseline leaderboard exists: {BASELINE_LB.exists()}')
print(f'Trained leaderboard exists:  {TRAINED_LB.exists()}')

## 1 · Baseline — pick best config from the hyperparameter sweep

Reads `baseline/leaderboard.json` (the hyperparameter sweep, `score_name=None` entries)
and selects the **overall best 2D pipeline+hyperparameters** across `end_to_end_2d` and
`ae_classifier_2d`.  That single config is then reused for every outcome score, giving a
fair apples-to-apples comparison (one fixed model for all scores, no per-score overfitting).

In [ ]:
configs = bcv.configs_from_hyper_leaderboard(
    HYPER_LB,
    scores=['QL2', 'PF2', 'CF', 'EF'],
)
print(f'\n{len(configs)} configs selected (one per outcome score).')

## 2 · Baseline — run 5-fold CV for k=2,3,4,5

Trains each selected config for every k ∈ {2,3,4,5} using StratifiedKFold(5).
Results are appended to `baseline/brainwear_baseline_cv.json`.  
Already-logged (pipeline, score, k) triples are skipped automatically.

In [ ]:

# Clear any stale quantile=False entries before running (the two QL2 entries
# in the existing JSON used quantile=False; re-run is needed with quantile=True).
import json as _json
if BASELINE_CV_LB.exists():
    stale = [r for r in _json.loads(BASELINE_CV_LB.read_text())
             if not r.get("config", {}).get("quantile", True)]
    if stale:
        print(f"Removing {len(stale)} stale quantile=False entries from {BASELINE_CV_LB.name}…")
        kept = [r for r in _json.loads(BASELINE_CV_LB.read_text()) if r not in stale]
        BASELINE_CV_LB.write_text(_json.dumps(kept, indent=2))
        print(f"  {len(kept)} entries remaining.")

# Skip if already logged; slow (GPU training) — run on HPC if needed
sweep = bcv.run_configs(
    configs,
    n_bins_range=(2, 3, 4, 5),
    n_folds=N_FOLDS,
    seed=SEED,
    lb_path=BASELINE_CV_LB,
)
if sweep:
    bcv.log_results(sweep, BASELINE_CV_LB)
else:
    print('All feasible entries already logged — nothing new to save.')


## 3 · Slot+AA-CBR — reused from existing sweep (no re-run)

Results come from `aacbr/leaderboard_trained.json`
(`eval_script='eval_trained_brainwear_2d'`), produced by the trained-model evaluation
notebook.  Nothing to recompute — quick sanity check only.

In [ ]:
_tr = [x for x in json.load(open(TRAINED_LB))
       if x.get('eval_script') == EVAL_SCRIPT
       and x.get('metrics', {}).get('cv_mean_f1') is not None]
print(f'Slot+AA-CBR entries in leaderboard_trained.json: {len(_tr)}')
for score in ('QL2', 'PF2', 'CF', 'EF'):
    for nb in (2, 3, 4, 5):
        cands = [x for x in _tr
                 if x['config'].get('score_name') == score
                 and x['config'].get('n_bins') == nb]
        if cands:
            b = max(cands, key=lambda x: x['metrics']['cv_mean_f1'])
            c = b['config']
            print(f'  {score} k={nb}: cv_F1={b["metrics"]["cv_mean_f1"]:.4f}  '
                  f'({c.get("char_model")}/{c.get("agg_mode")}/{c.get("strategy")})')

## 4 · Assemble + rank outcome scores

Assembles both leaderboards into a unified dict and ranks all 4 outcome scores by
Slot+AA-CBR skill (F1 − 1/k) at k=3 — a neutral anchor that avoids cherry-picking.

In [ ]:
import pandas as pd
importlib.reload(cmp_bw)

# ── Model filter ──────────────────────────────────────────────────────────────
# List the checkpoint folder names to include, or set to None to use all.
SLOT_CHECKPOINTS = ["brats_png_v14a_0.15_test", "brats_png_v17_weak_0.15_new_norm", "brats_png_v19a_entropy"]   # e.g. ["brats_png_v14a_gamma4", "brats_png_v20a_attn_skip"]

# Optional: rename the bracket label for each model (folder name → display name).
CHECKPOINT_LABELS = {"brats_png_v14a_0.15_test": "Fully supervised", 
                     "brats_png_v17_weak_0.15_new_norm": "Weakly supervised",
                     "brats_png_v19a_entropy": "Weakly supervised (spatial dice matching)"}   # e.g. {"brats_png_v14a_gamma4": "v14a"}
# ─────────────────────────────────────────────────────────────────────────────

data = cmp_bw.assemble(BASELINE_CV_LB, TRAINED_LB, EVAL_SCRIPT,
                       checkpoint_labels=CHECKPOINT_LABELS,
                       only_checkpoints=SLOT_CHECKPOINTS)

# Use the first Slot+AA-CBR method found for score ranking (avoids hardcoding the key).
slot_method = next((m for m in data if m.startswith('Slot+AA-CBR')), 'Slot+AA-CBR')
top4 = cmp_bw.rank_scores(data, method=slot_method, k=3)
print(f'Outcome scores ranked by {slot_method} skill at k=3: {top4}')

display(
    cmp_bw.to_dataframe(data, top4)
    .style
    .format('{:.3f}', na_rep='—')
    .background_gradient(cmap='RdYlGn', axis=None)
    .highlight_max(axis=0, subset=pd.IndexSlice[list(data.keys()), :],
                   props='font-weight:bold')
    .set_caption('Macro-F1 by method × score × k')
)

## 5 · Comparison figure (2×2 grid, all 4 scores)

2×2 subplots, one per outcome score in ranked order.  
x-axis = k ∈ {2,3,4,5}, lines = Baseline and Slot+AA-CBR, dashed = chance (1/k).  
Saved to `comparison/figures/compare_brainwear_png.pdf` and `.png`.

In [ ]:
from IPython.display import Image, display as ipy_display

for sub in ('figures', 'tables', 'results'):
    (FYP_ROOT / 'comparison' / sub).mkdir(parents=True, exist_ok=True)

NAME = 'compare_brainwear_png_weak_2'
stub = FYP_ROOT / 'comparison' / 'figures' / NAME
cmp_bw.make_figure(data, top4, 'BrainWear: QoL classification', str(stub))
ipy_display(Image(filename=str(stub) + '.png'))

## 6 · LaTeX table + analysis paragraph + CSV export

In [ ]:
TITLE = 'BrainWear 2D: QoL outcome-score classification'
TABLE_PATH    = FYP_ROOT / 'comparison' / 'tables' / f'{NAME}.tex'
ANALYSIS_PATH = FYP_ROOT / 'comparison' / 'tables' / f'analysis_{NAME}.tex'
CSV_PATH      = FYP_ROOT / 'comparison' / 'tables' / f'{NAME}.csv'
JSON_PATH     = FYP_ROOT / 'comparison' / 'results' / f'{NAME}.json'

cmp_bw.make_table(data, top4, TITLE, TABLE_PATH)
cmp_bw.make_analysis(data, top4, TITLE, ANALYSIS_PATH)
cmp_bw.write_csv(data, cmp_bw.SCORES, CSV_PATH)

JSON_PATH.write_text(json.dumps(
    {'title': TITLE, 'scores': top4, 'num_classes': cmp_bw.NUM_CLASSES, 'data': data}, indent=2))
print(f'json  → {JSON_PATH}')

print('\n===== LaTeX table =====')
print(TABLE_PATH.read_text())
print('\n===== LaTeX analysis =====')
print(ANALYSIS_PATH.read_text())

print('\n--- Best config per (method, score, k) ---')
display(cmp_bw.config_dataframe(data, top4))